In [9]:
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
#pd.set_option('display.width', 1000)  # display all columns without wrapping
#pd.set_option('display.max_columns', None)  # display all columns
#pd.set_option('display.max_rows', None)  # display all rows
pd.set_option('display.expand_frame_repr', False) # disable wrapping

In [ ]:
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor

def get_ext_price(symbol: str, include_extended_hours: bool = True) -> float:
	ticker = yf.Ticker(symbol)

	if include_extended_hours:
		info = ticker.info or {}
		# Prefer explicit post-market and pre-market quotes when available.
		post_market_price = info.get("postMarketPrice")
		if post_market_price is not None:
			return round(float(post_market_price), 2)

		pre_market_price = info.get("preMarketPrice")
		if pre_market_price is not None:
			return round(float(pre_market_price), 2)

		# Fallback for extended hours: latest intraday trade including pre/post data.
		intraday = ticker.history(period="1d", interval="1m", prepost=True)
		if not intraday.empty:
			close_series = intraday["Close"].dropna()
			if not close_series.empty:
				return round(float(close_series.iloc[-1]), 2)

	# Try the most up-to-date value first.
	fast_info = ticker.fast_info or {}
	last_price = fast_info.get("lastPrice")
	if last_price is not None:
		return round(float(last_price), 2)

	# Fallback: use latest close from recent history.
	history = ticker.history(period="5d")
	if history.empty:
		raise RuntimeError(f"Unable to fetch {symbol} price from yfinance")
	return round(float(history["Close"].iloc[-1]), 2)

def get_prices(tickers, include_extended_hours: bool = True) -> dict:
	"""Fetch prices for unique tickers concurrently; failures map to NaN."""
	def safe_get(symbol):
		try:
			return get_ext_price(symbol, include_extended_hours)
		except Exception as e:
			print(symbol, e)
			return float('nan')
	unique = list(dict.fromkeys(tickers))
	with ThreadPoolExecutor(max_workers=16) as ex:
		prices = list(ex.map(safe_get, unique))
	return dict(zip(unique, prices))
#print(get_ext_price("AAPL")) #testing   

In [ ]:
df = pd.read_csv(r'E:\My Drive\tax\out_fidelity.csv', on_bad_lines='skip')

df = df[df['Option'].isna() & ~df['Ticker'].isin(['sell', 'buy'])] # .head(10) Limit to 10 rows for testing, rows Option column empty and not buy or sell
#print(df[['Ticker', 'Option','OptionCnt', 'sell', 'CostBasisPerShare','buy']]) #testing
df['o_CurPrice'] = df['Ticker'].map(get_prices(df['Ticker'], include_extended_hours=False)) # fast_info only, each unique ticker once, in parallel
df['o_ref_price'] = df['sell'].combine_first(pd.to_numeric(df['CostBasisPerShare'], errors='coerce').abs())
#print(df[['Ticker', 'sell', 'CostBasisPerShare','o_CurPrice', 'o_ref_price']]) #testing

df['o_ref_price'] = pd.to_numeric(df['o_ref_price'], errors='coerce')
df['o_Diff'] = df['o_CurPrice'] - df['o_ref_price']

df = df.sort_values('o_Diff', ascending=False)
df['sell'] = df['sell'].fillna('')
df['OptionCnt'] = df['OptionCnt'].apply(lambda x: int(x) if pd.notna(x) else '')
print(df[['Account', 'Ticker', 'CostBasisPerShare', 'sell', 'o_CurPrice', 'o_Diff', 'OptionCnt']])

In [ ]:
import datetime as dt
import re
from datetime import date
from concurrent.futures import ThreadPoolExecutor
import yfinance as yf

def calc_rate(cost, symbol_to):
    if not symbol_to.strip():
        return
    symbolOp = re.split(r'([\d.]+)', symbol_to)
    final_value = float(symbolOp[3])
    final_date = dt.datetime.strptime(str(int(symbolOp[1])), '%y%m%d').date()
    days_hold = (final_date - date.today()).days+1
    rate_of_gain = float(cost) / final_value * (365 / days_hold) * 100
    return round(rate_of_gain)

def _fetch_chains(keys):
    """Fetch option chains for unique (ticker, expiry) pairs concurrently."""
    def fetch(key):
        try:
            return key, yf.Ticker(key[0]).option_chain(key[1])
        except Exception as e:
            print(key[0], e)
            return key, None
    with ThreadPoolExecutor(max_workers=16) as ex:
        return dict(ex.map(fetch, keys))

def _scan(df, opt_type):
    # Parse all symbols first so each (ticker, expiry) chain is downloaded only once.
    parsed = {}
    for i in df.index:
        try:
            symbolOp = re.split(r'([\d.]+)', df['Symbol'][i])
            myDate = dt.datetime.strptime(str(int(symbolOp[1])), '%y%m%d').date()
            parsed[i] = (symbolOp[0], myDate.strftime('%Y-%m-%d'), float(symbolOp[3]))
        except Exception as e:
            print(df['Symbol'][i], e)
    chains = _fetch_chains({(tkr, expiry) for tkr, expiry, _ in parsed.values()})
    for i, (tkr, expiry, strike) in parsed.items():
        try:
            optC = chains[(tkr, expiry)]
            opt = optC.puts if opt_type == 'P' else optC.calls
            opt = opt[ (opt['strike'] - strike).abs() < 1 ]
            df.loc[i,'o_Buy'] = pd.Series(opt['bid']).values[0]
            sell = float(df['sell'][i])
            df.loc[i,'o_GainLoss'] = (df['o_Buy'][i] - sell) * df['Quantity'][i]
            df.loc[i,'o_percent'] = round(df['o_GainLoss'][i] * 100 / sell / abs(df['Quantity'][i]))
            df.loc[i, 'o_rate'] = calc_rate(df['o_Buy'][i], df['Symbol'][i])
        except Exception as e:
            print(tkr, e)
    df['o_percent'] = pd.to_numeric(df['o_percent'], errors='coerce').astype('Int64')
    df['o_rate'] = pd.to_numeric(df['o_rate'], errors='coerce').astype('Int64')
    return df[['Account', 'Symbol', 'o_Buy', 'sell', 'o_GainLoss', 'o_percent', 'o_rate']]

In [ ]:
def _scan_P(df):
    return _scan(df, 'P')

In [14]:
pd.options.mode.chained_assignment = None  # default='warn'
pd.set_option('display.width', 1000)  # display all columns without wrapping
pd.set_option('display.max_columns', None)  # display all columns
pd.set_option('display.max_rows', None)  # display all rows

df = pd.read_csv(r'E:\My Drive\tax\out_fidelity.csv', on_bad_lines='skip')
df = df[(df['Option'] == 'P') & (~df['Ticker'].isin(['sell', 'buy']))]
#print(df[['Account', 'Symbol', 'sell']]) #testing
df_P = _scan_P(df)
df_P = df_P.sort_values(by=['o_percent'], ascending=False)
print(df_P)

         Account            Symbol   o_Buy   sell  o_GainLoss  o_percent  o_rate
58     218320886    LITE260807P625    0.60   50.0       49.40         99       9
60     X83939133    MRVL260821P185    5.50   19.6       14.10         72      60
123    X83939133  TSLA260821P307.5    5.75   15.2        9.45         62      38
17     414745529     ASTS260821P65    5.20   10.9        5.70         52     162
42     X83939133   CRWV260821P87.5    7.55   12.6        5.05         40     175
39     X83939133     CRCL260821P65    6.70    9.6        2.90         30     209
104  Rosetta4260    SPCX260821P115   11.10   13.8        2.70         20     196
105    X83939133    SPCX260821P116   11.60   13.2        1.60         12     203
77     218320885    ORCL260821P155   14.85   14.5       -0.35         -2     194
101    X83939133   SNDK260821P1720  380.80  300.0      -80.80        -27     449
54     414745529     IONQ260821P50   10.15    7.0       -3.15        -45     412
106    X83939133    SPCX2608

In [ ]:
def _scan_C(df):
    return _scan(df, 'C')

In [16]:
pd.set_option('display.width', 1000)  # display all columns without wrapping
df = pd.read_csv(r'E:\My Drive\tax\out_fidelity.csv', on_bad_lines='skip')
df = df[(df['Option'] == 'C') & (~df['Ticker'].isin(['sell', 'buy']))]
# #df = df[~df['Symbol'].str.contains('260515')] # filter rows, keep only current month call 
# #df = df[df['Account'] == 'X65750304'] # only rows on account   
df_C = _scan_C(df)
df_C = df_C.sort_values(by=['o_percent'], ascending=False)
print(df_C)
# #df_C.reset_index(drop=True)

       Account           Symbol   o_Buy   sell  o_GainLoss  o_percent  o_rate
116  414745529    SQQQ260821C47    0.30   4.15        3.85         93      13
32   X65750304   COIN260821C175    2.44  17.15       14.71         86      28
110  218320886   SPCX260821C170    2.91  20.00       17.09         85      35
31   X65750304   COIN260821C170    3.20  19.90       16.70         84      38
109  218320886   SPCX260821C165    3.30  19.10       15.80         83      41
115  231736622  SQQQ260821C44.5    0.75   4.05        3.30         81      34
114  233186075  SQQQ260821C43.5    0.86   4.00        3.14         78      40
100  218320886    SMCI260821C35    1.44   5.25        3.81         73      83
35   218320886   COST260821C970   10.05  26.00       15.95         61      21
129  X83939133    UBER260821C75    1.83   4.75        2.92         61      49
9    X65750304     AGQ260821C70    2.70   6.80        4.10         60      78
72   X83939133   NVDA260821C215    5.00   9.37        8.74      